# External model — chat completion

OpenAI-compatible **`POST …/v1/chat/completions`** on the gateway. One JSON request and one JSON response (**`choices[0].message.content`**).

**Prereq:** Python 3.9+ (stdlib: `json`, `ssl`, `urllib`).

**Credentials:** Same pattern as the no-key validation notebook: `MAAS_API_KEY` / `API_KEY` / **Demo quick swap** (mint with `POST …/maas-api/v1/api-keys` and subscription **`external-models-subscription`** when you use that flow). **`VERIFY_TLS`** from the environment (`1` / `true` for strict TLS).

**Two presets:** GPT-4o vs Claude. Set **`ACTIVE_EXTERNAL`** in **Demo quick swap** to **`"gpt4o"`** or **`"claude"`**.

## Demo quick swap

Set **`ACTIVE_EXTERNAL`** (and optionally **`DEMO_API_KEY`** between takes). Run this cell, then **Load presets**.

| Variable | Effect |
|----------|--------|
| `ACTIVE_EXTERNAL` | **`"gpt4o"`** or **`"claude"`** — which preset to use. |
| `DEMO_API_KEY` | Non-empty → overrides `MAAS_API_KEY` / `API_KEY` env. |

In [ ]:
ACTIVE_EXTERNAL = "claude"  # or "gpt4o"
DEMO_API_KEY = ""

## Load presets

**HOST**, routes, and model ids are fixed — only **`ACTIVE_EXTERNAL`** selects GPT-4o vs Claude.

In [ ]:
import json
import os
import ssl
import urllib.error
import urllib.request

HOST = "maas.apps.rosa.jland.li5b.p3.openshiftapps.com"

_PRESET_GPT4O = {
    "url": f"https://{HOST}/gpt-4o/v1/chat/completions",
    "model": "gpt-4o",
}

_PRESET_CLAUDE = {
    "url": f"https://{HOST}/claude-sonnet-4-20250514/v1/chat/completions",
    "model": "claude-sonnet-4-20250514",
}

_presets = {"gpt4o": _PRESET_GPT4O, "claude": _PRESET_CLAUDE}
_ae = globals().get("ACTIVE_EXTERNAL", "gpt4o")
ACTIVE_EXTERNAL = _ae.strip() if isinstance(_ae, str) and _ae.strip() else "gpt4o"
if ACTIVE_EXTERNAL not in _presets:
    raise SystemExit('ACTIVE_EXTERNAL must be "gpt4o" or "claude"')
_sel = _presets[ACTIVE_EXTERNAL]
EXTERNAL_CHAT_COMPLETIONS_URL = _sel["url"]
EXTERNAL_MODEL_NAME = _sel["model"]

_ak = globals().get("DEMO_API_KEY", "")
if isinstance(_ak, str) and _ak.strip():
    API_KEY = _ak.strip()
else:
    API_KEY = (os.environ.get("MAAS_API_KEY") or os.environ.get("API_KEY") or "").strip()

VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")

if not API_KEY:
    raise SystemExit("Set DEMO_API_KEY or MAAS_API_KEY / API_KEY.")

print("ACTIVE_EXTERNAL :", ACTIVE_EXTERNAL)
print("Model id          :", EXTERNAL_MODEL_NAME)
print("POST URL          :", EXTERNAL_CHAT_COMPLETIONS_URL)
print("VERIFY_TLS        :", VERIFY_TLS)
print("API key set       :", bool(API_KEY))

## Chat completion

**`POST`** non-streaming JSON (no `stream` flag). Parses **`choices[0].message.content`** as a string, or concatenates simple text blocks if **`content`** is a list.

In [ ]:
USER_MESSAGE = "What model am I using?"
MAX_TOKENS = 256


def _ssl_ctx():
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    return ctx


def _assistant_text(obj: dict) -> str:
    choices = obj.get("choices") or []
    if not choices:
        return ""
    msg = (choices[0].get("message") or {}) if isinstance(choices[0], dict) else {}
    c = msg.get("content")
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        parts = []
        for b in c:
            if isinstance(b, dict):
                t = b.get("text") or b.get("content")
                if isinstance(t, str):
                    parts.append(t)
            elif isinstance(b, str):
                parts.append(b)
        return "".join(parts)
    return ""


def chat_completion(url: str, *, api_key: str, model: str, user_message: str, max_tokens: int):
    """POST /v1/chat/completions; return (http_status, error_str_or_none)."""
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": user_message}],
        "max_tokens": max_tokens,
    }
    body = json.dumps(payload).encode("utf-8")
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    req = urllib.request.Request(url, data=body, headers=headers, method="POST")
    ctx = _ssl_ctx()
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=300) as resp:
            status = resp.getcode()
            raw = resp.read().decode("utf-8", errors="replace")
    except urllib.error.HTTPError as e:
        err = e.read().decode("utf-8", errors="replace")
        print("HTTP", e.code, err[:2000])
        return e.code, err
    except urllib.error.URLError as e:
        msg = str(e.reason) if getattr(e, "reason", None) else str(e)
        print("Error:", msg)
        return None, msg
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        print("Non-JSON response:", raw[:2000])
        return status, "invalid JSON"
    text = _assistant_text(obj) if isinstance(obj, dict) else ""
    if text:
        print(text)
    else:
        print("(no choices[0].message.content)", raw[:1500])
    u = obj.get("usage") if isinstance(obj, dict) else None
    if u:
        print("usage:", u)
    return status, None


http_status, req_err = chat_completion(
    EXTERNAL_CHAT_COMPLETIONS_URL,
    api_key=API_KEY,
    model=EXTERNAL_MODEL_NAME,
    user_message=USER_MESSAGE,
    max_tokens=MAX_TOKENS,
)
print("HTTP", http_status, "|", "ok" if not req_err else req_err[:200])